<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/gemma_lora-json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [2]:
%pip install -U -q keras-hub keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.27.1 which is incompatible.


In [3]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [4]:
import keras
import keras_hub

In [5]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")

In [6]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

In [7]:
prompt = template.format(
  instruction='I have been having headaches every day for the past week with sensitivity to light.',
  response=''
)

print(gemma_lm.generate(prompt, max_length=200))

Instruction:
I have been having headaches every day for the past week with sensitivity to light.

Response:
Okay, I understand you've been experiencing headaches daily with sensitivity to light for the past week.  It's important to note that headaches and light sensitivity can be symptoms of several different conditions.  Here's a breakdown of potential causes:

*   **Migraines:** These are often characterized by throbbing pain, sensitivity to light and sound, and sometimes nausea.
*   **Tension Headaches:** These are the most common type of headache, often described as a tight band around the head.
*   **Light Sensitivity (Photophobia):** This is a common symptom, where you experience discomfort or pain when exposed to bright light.
*   **Cluster Headaches:** These are less frequent but can be severe and involve intense pain around one eye.
*   **Other potential causes:**  It's also possible that there could be other


In [8]:
from datasets import load_dataset
import json

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
df = ds.to_pandas().sample(500, random_state=42)

def to_json_response(response):
  return json.dumps({
    "assessment": response[:200],  # truncate long responses
    "recommendations": ["Consult a doctor", "Monitor symptoms"],
    "urgency": "medium"
  })

prompt_template = "Instruction:\n{instruction}\n\nResponse:\n"

features = {
  "prompts": [prompt_template.format(instruction=p) for p in df["input"].tolist()],
  "responses": [to_json_response(r) for r in df["output"].tolist()]
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

In [9]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

500
Instruction:
I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.

Response:

{"assessment": "Dear patient Here are the possibilities of what you might have.1)PhlebitisPhlebitis means inflammation of the veins, and can cause redness, itching, irritation, pain, and swelling. A simple Doppler ca", "recommendations": ["Consult a doctor", "Monitor symptoms"], "urgency": "medium"}


In [10]:
gemma_lm.backbone.enable_lora(rank=4)

In [11]:
# Limit the input sequence length to 256 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 256
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

In [12]:
gemma_lm.fit(features, epochs=1, batch_size=1)

500/500 ━━━━━━━━━━━━━━━━━━━━ 205s 349ms/step - loss: 0.7481 - sparse_categorical_accuracy: 0.5152


In [13]:
prompt = template.format(
  instruction="I have been having headaches every day for the past week with sensitivity to light.",
  response=''
)
print(gemma_lm.generate(prompt, max_length=200))

Instruction:
I have been having headaches every day for the past week with sensitivity to light.

Response:
{"assessment": "Hi, I understand you're experiencing headaches with sensitivity to light. It's important to note that headaches can be caused by many things, including tension headaches, migraines, and sinus infections. However, sensitivity to light can be a symptom of a few conditions, such as migraine, and can be caused by a number of factors. I would recommend that you consult a doctor to determine the cause of your headaches. I am unable to provide medical advice. I am an AI Chatbot. I am here to provide information and answer your questions. I am not able to provide medical advice. I am unable to provide medical advice. I am unable to provide medical advice. I am unable to provide medical advice. I am unable to provide medical advice. I am unable to provide medical advice. I am unable to provide medical advice. I am unable to provide medical


In [14]:
gemma_lm.backbone.save_lora_weights("lora_weights.lora.h5")

In [16]:
from huggingface_hub import notebook_login
notebook_login()

In [17]:
from huggingface_hub import HfApi

api = HfApi()

# Create a repo and upload
api.create_repo("busycaesar/gemma3-healthcaremagic-json-lora", exist_ok=True)

api.upload_file(
    path_or_fileobj="lora_weights.lora.h5",
    path_in_repo="lora_weights.lora.h5",
    repo_id="busycaesar/gemma3-healthcaremagic-json-lora"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  lora_weights.lora.h5        :  21%|##1       |  586kB / 2.76MB            

CommitInfo(commit_url='https://huggingface.co/busycaesar/gemma3-healthcaremagic-json-lora/commit/88b7c68e04842b77dac8c8f2f24460a7a62124d0', commit_message='Upload lora_weights.lora.h5 with huggingface_hub', commit_description='', oid='88b7c68e04842b77dac8c8f2f24460a7a62124d0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/busycaesar/gemma3-healthcaremagic-json-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='busycaesar/gemma3-healthcaremagic-json-lora'), pr_revision=None, pr_num=None)